# PAX title reconciliation and BGG-ID review

This notebook documents the manual review and validation of PAX title records against BoardGameGeek (BGG) records.

The canonical matching workflow is now implemented by:

- `scripts/pax_labeling/resolve_existing_pax_titles.py`
- `scripts/pax_labeling/finalize_pax_title_corrections.py`

This notebook is a review and diagnostics companion. It preserves the retained manual BGG-ID corrections, shows how exact matching behaves, and inspects the resolver/finalizer outputs. It does not replace the scripts or silently update source data.

## Intended run order

Run the resolver and finalizer first when reviewing a new raw title export. Then run this notebook from the repository so the diagnostics and review tables use the same reference data. The final East-library export is optional.

The notebook expects the working directory to be either the repository root or `data_pipeline/notebooks/`. It reads source/reference CSVs from `data/` and only writes the explicitly named PAX export in the final section.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.options.display.max_columns = 100
pd.options.display.max_rows = 100


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir() and (candidate / "scripts").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repository root from the notebook working directory.")


REPO_ROOT = find_repo_root(Path.cwd())
DATA_ROOT = REPO_ROOT / "data"
PAX_ROOT = DATA_ROOT / "pax"
INGEST_ROOT = DATA_ROOT / "ingest"
PROCESSED_ROOT = DATA_ROOT / "transform" / "processed" / "1774130397"

print(f"Repository root: {REPO_ROOT}")

Repository root: /home/msnow/git/pax_tt_recommender


## Load source and BGG reference data

The raw PAX export is the input being corrected. BGG ranks provide canonical names and years; first-publisher and all-publisher extracts provide publisher evidence for diagnostics and review. The current run loaded 175,041 BGG reference rows and 11,418 raw PAX title rows.

In [2]:
df_all_pubs = pd.read_csv(
    PROCESSED_ROOT / "processed_games_boardgamepublisher_1774130397.csv",
    sep="|",
    escapechar="\\",
)
df_publishers = pd.read_csv(INGEST_ROOT / "game_data" / "first_publishers.csv").rename(
    columns={"id": "bgg_id"}
)
df_bgg = (
    pd.read_csv(
        INGEST_ROOT / "ranks" / "boardgame_ranks_20260319.csv",
        sep="|",
        escapechar="\\",
    )
    .rename(columns={"id": "bgg_id", "name": "bgg_name"})
    .merge(df_publishers, on="bgg_id", how="left")
)

df_paxtt = pd.read_csv(PAX_ROOT / "title_corrections_raw.csv", sep="|", encoding="cp1252")
df_paxtt["bgg_id"] = pd.to_numeric(df_paxtt["bgg_id"], errors="coerce").astype("Int64")
df_paxtt.loc[df_paxtt["bgg_id"].eq(0), "bgg_id"] = pd.NA

print(f"BGG reference rows: {len(df_bgg):,}")
print(f"Raw PAX title rows: {len(df_paxtt):,}")
display(df_paxtt.head())

BGG reference rows: 175,041
Raw PAX title rows: 11,418


,title,title_id,game,game_barcode,publisher_name,bgg_id,game_year,status
0,"""A Matter of Honor, Sire""",1148,2680,TTL1019O,AEG - DO NOT USE,<NA>,2014,1
1,#artbox,5025,10440,TTL06494I,Lavka Games,<NA>,2024,2
2,1 A.M. Jailbreak,5189,10805,25EAST0418,Saashi & Saashi,<NA>,2025,0
3,1 A.M. Jailbreak,5189,10806,25EAST0414,Saashi & Saashi,<NA>,2025,0
4,1 A.M. Jailbreak,5189,10807,25EAST0410,Saashi & Saashi,<NA>,2025,0


## Apply retained manual BGG-ID corrections

The retained manual corrections are represented here as explicit tables so a reviewer can inspect them as data.

The existing-BGG-ID corrections run first, followed by title-ID corrections. Title ID `269` resolves to `72125` in the retained correction table. This run changed 15 rows through existing-BGG-ID overrides and 259 rows through title-ID overrides, representing 132 unique title-ID overrides. Durable future overrides should be moved into the resolver's `--manual-fixes` CSV rather than added as new notebook cells.

In [3]:
MANUAL_BGG_ID_FIXES = {
    89862: 55253,
    234497: 242409,
    268191: 244271,
    224965: 72376,
    63993: 2290,
    282031: 364358,
}

existing_bgg_override = df_paxtt["bgg_id"].map(MANUAL_BGG_ID_FIXES)
existing_bgg_mask = existing_bgg_override.notna()
df_paxtt.loc[existing_bgg_mask, "bgg_id"] = existing_bgg_override.loc[existing_bgg_mask].astype("Int64")

print(f"Rows changed by existing-BGG-ID overrides: {int(existing_bgg_mask.sum()):,}")

Rows changed by existing-BGG-ID overrides: 15


In [4]:
TITLE_ID_FIXES_1 = {
    55: 55253,
    2561: 242409,
    2588: 244271,
    3573: 72376,
    3552: 2290,
    3321: 364358,
    387: 224,
    2443: 224,
    423: 34010,
    3830: 324246,
    1197: 20437,
    3601: 20437,
    1544: 183231,
    4311: 357563,
    4341: 370391,
    4146: 359402,
    992: 12248,
    3360: 317983,
    3858: 317983,
    5019: 420214,
    1912: 207208,
    5208: 419454,
    49: 69789,
    4649: 348554,
    2738: 446,
    60: 10093,
    5164: 418354,
    5038: 418354,
    66: 43231,
    73: 551,
    75: 760,
    5280: 402674,
    2023: 2425,
    5532: 446748,
    5192: 424730,
    5184: 421310,
    5216: 420805,
    4301: 367891,
    2146: 231824,
    4506: 366994,
}

In [5]:
TITLE_ID_FIXES_2 = {
    174: 1294,
    3693: 329450,
    1205: 131121,
    5403: 395623,
    2080: 200058,
    2503: 316093,
    3673: 316093,
    5122: 400265,
    4737: 3076,
    755: 37196,
    4422: 147020,
    3966: 84876,
    5155: 552,
    3879: 290236,
    3269: 170583,
    148: 65521,
    167: 478,
    1857: 478,
    5497: 205398,
    2715: 1294,
    2720: 1294,
    1306: 145639,
    181: 24742,
    5135: 394887,
    1925: 176306,
    5239: 424435,
    5483: 396895,
    194: 39463,
    1093: 121414,
    1503: 131357,
    3857: 317981,
    2148: 225694,
    4582: 376747,
    4457: 376747,
    3803: 317457,
    241: 6137,
    243: 121,
    5041: 181329,
    5205: 181329,
    269: 72125,
}

In [6]:
TITLE_ID_FIXES_3 = {
    283: 33160,
    3110: 131121,
    3588: 329450,
    5010: 379,
    4985: 385761,
    5042: 439209,
    300: 40628,
    4988: 40628,
    5298: 419195,
    5043: 374024,
    4329: 376218,
    5302: 417518,
    3368: 275969,
    5185: 403683,
    2818: 182791,
    5487: 456566,
    196: 66587,
    5242: 380109,
    5250: 419639,
    5199: 418858,
    4306: 368103,
    946: 12830,
    3018: 245598,
    3174: 198328,
    2725: 218,
    2894: 246147,
    3466: 172242,
    3115: 159517,
    1502: 129622,
    1583: 137269,
    2353: 1463,
    1112: 10167,
    5500: 440257,
    5138: 407317,
    5136: 407318,
    5406: 436590,
    5516: 208134,
    4405: 244271,
    5305: 407823,
    5238: 422478,
}

In [7]:
TITLE_ID_FIXES_4 = {
    5301: 370963,
    2817: 233867,
    5525: 205708,
    5071: 422693,
    2903: 166859,
    4556: 358000,
    5286: 343844,
    4379: 345972,
    4879: 417317,
    4436: 332686,
    5053: 192074,
    3631: 329845,
}

In [8]:
TITLE_ID_FIXES = {}
for fixes in [TITLE_ID_FIXES_1, TITLE_ID_FIXES_2, TITLE_ID_FIXES_3, TITLE_ID_FIXES_4]:
    TITLE_ID_FIXES.update(fixes)

title_override = df_paxtt["title_id"].map(TITLE_ID_FIXES)
title_override_mask = title_override.notna()
df_paxtt.loc[title_override_mask, "bgg_id"] = title_override.loc[title_override_mask].astype("Int64")

print(f"Rows changed by title-ID overrides: {int(title_override_mask.sum()):,}")
print(f"Unique retained title-ID overrides: {len(TITLE_ID_FIXES):,}")

Rows changed by title-ID overrides: 259
Unique retained title-ID overrides: 132


## Diagnostic matching: existing BGG IDs

This section measures how many raw rows already resolve through their BGG ID after the retained manual corrections. It is a coverage diagnostic, not the canonical resolver.

In [9]:
df_m1 = df_paxtt.merge(
    df_bgg[["bgg_name", "bgg_id"]],
    on="bgg_id",
    how="inner",
)
print(f"Rows matched by BGG ID: {len(df_m1):,}")
display(df_m1[["title", "title_id", "bgg_id", "bgg_name"]].head(10))

Rows matched by BGG ID: 4,085


,title,title_id,bgg_id,bgg_name
0,12 Rivers,4341,370391,12 Rivers
1,1920 Wall Street,1989,226255,1920 Wall Street
2,1942 USS Yorktown,3378,279307,1942 USS Yorktown
3,1942 USS Yorktown,3378,279307,1942 USS Yorktown
4,1942 USS Yorktown,3378,279307,1942 USS Yorktown
5,1987 Channel Tunnel,3379,279306,1987 Channel Tunnel
6,1987 Channel Tunnel,3379,279306,1987 Channel Tunnel
7,1987 Channel Tunnel,3379,279306,1987 Channel Tunnel
8,3 Laws of Robotics,3324,271447,3 Laws of Robotics
9,5-Minute Marvel,3191,253618,5-Minute Marvel


The current run matched 4,085 rows through the BGG-ID join. Rows not represented here continue to the title/publisher/year diagnostics and, in the current workflow, to the resolver's broader candidate scoring.

## Diagnostic matching: exact title, publisher, and year

This diagnostic branch tests a stricter exact-title/publisher match with a one-year tolerance. It is useful for explaining ambiguity, but it does not decide final IDs because duplicate titles and publisher variants can produce multiple candidates.

In [10]:
unmatched_title_ids = ~df_paxtt["title_id"].isin(df_m1["title_id"])
df_m2 = df_paxtt.loc[
    unmatched_title_ids,
    ["title", "title_id", "game_barcode", "publisher_name", "game_year"],
].drop_duplicates()

df_m2["title_lower"] = df_m2["title"].str.lower()
df_m2["publisher_lower"] = df_m2["publisher_name"].str.lower()
df_bgg["title_lower"] = df_bgg["bgg_name"].str.lower()
df_bgg["publisher_lower"] = df_bgg["publisher"].str.lower()

df_m2 = df_m2.merge(
    df_bgg[["bgg_id", "title_lower", "usersrated", "yearpublished", "publisher_lower"]],
    on=["title_lower", "publisher_lower"],
    how="inner",
)
df_m2 = df_m2.loc[df_m2["game_year"] + 1 >= df_m2["yearpublished"]]
df_m2 = df_m2.drop(columns=["title_lower", "publisher_lower"])

duplicate_bgg_ids_m2 = (
    df_m2[["title", "bgg_id"]]
    .drop_duplicates()
    .loc[lambda frame: frame["bgg_id"].duplicated(), "bgg_id"]
    .tolist()
)
duplicate_titles_m2 = (
    df_m2[["title", "bgg_id"]]
    .drop_duplicates()
    .loc[lambda frame: frame["title"].duplicated(), "title"]
    .tolist()
)

print(f"Exact title/publisher/year rows: {len(df_m2):,}")
print(f"Ambiguous BGG IDs: {len(duplicate_bgg_ids_m2):,}")
print(f"Ambiguous titles: {len(duplicate_titles_m2):,}")
display(df_m2.head())

Exact title/publisher/year rows: 2,644
Ambiguous BGG IDs: 23
Ambiguous titles: 0


,title,title_id,game_barcode,publisher_name,game_year,bgg_id,usersrated,yearpublished
0,1 A.M. Jailbreak,5189,25EAST0418,Saashi & Saashi,2025,428224,380,2024
1,1 A.M. Jailbreak,5189,25EAST0414,Saashi & Saashi,2025,428224,380,2024
2,1 A.M. Jailbreak,5189,25EAST0410,Saashi & Saashi,2025,428224,380,2024
3,1001 Islands,4036,TTL16795I,Ludonaute,2022,351972,199,2022
4,12 Days,5371,25UNPL0061,Calliope Games,2025,130899,1640,2011


The current run produced 2,644 rows here, with 23 repeated BGG IDs and no repeated titles. Those repeated candidates are why this branch remains diagnostic rather than an automatic correction rule.

## Diagnostic matching: exact title and year

This broader branch drops publisher matching to show where title/year evidence is insufficient on its own. It remains diagnostic and should not be interpreted as a final assignment method.

In [11]:
matched_title_ids = set(df_m1["title_id"]) | set(df_m2["title_id"])
df_m3 = df_paxtt.loc[
    ~df_paxtt["title_id"].isin(matched_title_ids),
    ["title", "title_id", "game_barcode", "publisher_name", "game_year"],
].drop_duplicates()

df_m3["title_lower"] = df_m3["title"].str.lower()
df_bgg["title_lower"] = df_bgg["bgg_name"].str.lower()
df_m3 = df_m3.merge(
    df_bgg[["bgg_name", "bgg_id", "title_lower", "usersrated", "yearpublished", "publisher"]],
    on="title_lower",
    how="inner",
)
df_m3 = df_m3.loc[df_m3["game_year"] + 1 >= df_m3["yearpublished"]]

duplicate_bgg_ids_m3 = (
    df_m3[["title", "bgg_id"]]
    .drop_duplicates()
    .loc[lambda frame: frame["bgg_id"].duplicated(), "bgg_id"]
    .tolist()
)
duplicate_titles_m3 = (
    df_m3[["title", "bgg_id"]]
    .drop_duplicates()
    .loc[lambda frame: frame["title"].duplicated(), "title"]
    .tolist()
)

print(f"Exact title/year rows: {len(df_m3):,}")
print(f"Ambiguous BGG IDs: {len(duplicate_bgg_ids_m3):,}")
print(f"Ambiguous titles: {len(duplicate_titles_m3):,}")
display(df_m3.head())

Exact title/year rows: 3,322
Ambiguous BGG IDs: 23
Ambiguous titles: 193


,title,title_id,game_barcode,publisher_name,game_year,title_lower,bgg_name,bgg_id,usersrated,yearpublished,publisher
0,"""A Matter of Honor, Sire""",1148,TTL1019O,AEG - DO NOT USE,2014,"""a matter of honor, sire""","""A Matter of Honor, Sire""",153128,22,2013,Against the Odds
1,10 days in asia,1,TTL100RV,Out of the Box,2014,10 days in asia,10 Days in Asia,22398,1204,2007,Out of the Box Publishing
2,1313 Dead End Drive,2053,TTL04115I,Parker Brothers,2017,1313 dead end drive,1313 Dead End Drive,4953,981,2002,Milton Bradley
3,1775: Rebellion,1946,TTL03505I,Academy Games,2017,1775: rebellion,1775: Rebellion,128996,4468,2013,"Academy Games, Inc."
4,1775: Rebellion,1946,TTL03505I,Academy Games,2018,1775: rebellion,1775: Rebellion,128996,4468,2013,"Academy Games, Inc."


The current run produced 3,322 exact-title/year rows, with 23 repeated BGG IDs and 193 repeated titles. This confirms that title-only matching needs publisher, year, prior, and review evidence.

## Review notes

The review process records a small set of unresolved or intentionally protected records, but does not encode their semantics as executable rules:

- Do not alter names: game ID `2951`.
- Not in BGG: game IDs `3021`, `3337`, and `3762`.
- Additional spot-check IDs: `14`, `2328`, `5`, and `1971`.

Keep these as review notes until their meaning is confirmed. Do not turn them into automatic exclusions or name-preservation rules without confirming their intended meaning.

## Review current resolver output

The resolver is the current path for existing PAX titles. In this run it produced 11,418 rows: 9,382 high-confidence, 1,259 medium-confidence, 466 low-confidence, 288 manual, 21 excluded, and 2 unresolved. It also produced 523 review-queue rows. The 7,607 differing BGG IDs include 7,522 fills for previously missing IDs, 62 changes between two present IDs, and 4 removals; 19 rows remain missing on both sides. Inspect confidence, changed IDs, and the generated top-three review queue before finalization. If its output does not exist, run `scripts/pax_labeling/resolve_existing_pax_titles.py` first.

In [12]:
MATCH_PATH = PAX_ROOT / "title_corrections_with_new_bgg_id.csv"
REVIEW_PATH = PAX_ROOT / "title_corrections_review_top3.csv"

if MATCH_PATH.exists():
    df_match = pd.read_csv(MATCH_PATH)
    print(f"Resolver rows: {len(df_match):,}")
    display(
        df_match.groupby("match_confidence", dropna=False)
        .size()
        .rename("rows")
        .to_frame()
    )
    changed_ids = df_match[
        pd.to_numeric(df_match["bgg_id"], errors="coerce")
        != pd.to_numeric(df_match["new_bgg_id"], errors="coerce")
    ]
    print(f"Rows with changed BGG IDs: {len(changed_ids):,}")
    display(changed_ids.head(50))
else:
    df_match = pd.DataFrame()
    print(f"Resolver output not found: {MATCH_PATH}")

Resolver rows: 11,418


,rows
match_confidence,
excluded,21
high,9382
low,466
manual,288
medium,1259
unresolved,2


Rows with changed BGG IDs: 7,607


,title,title_id,game,game_barcode,publisher_name,bgg_id,game_year,status,original_bgg_name,new_bgg_id,new_bgg_name,match_confidence,match_reason,match_score_margin,top_candidate_count,excluded_title_id,used_publisher_first_fallback
0,"""A Matter of Honor, Sire""",1148,2680,TTL1019O,AEG - DO NOT USE,NaN,2014,1,NaN,153128.0,"""A Matter of Honor, Sire""",medium,scored_candidate,2228.932000,9,False,False
1,#artbox,5025,10440,TTL06494I,Lavka Games,NaN,2024,2,NaN,280304.0,ARTBOX,high,scored_candidate,9999.000000,1,False,False
2,1 A.M. Jailbreak,5189,10805,25EAST0418,Saashi & Saashi,NaN,2025,0,NaN,428224.0,1 A.M. Jailbreak,high,scored_candidate,5529.353000,25,False,False
3,1 A.M. Jailbreak,5189,10806,25EAST0414,Saashi & Saashi,NaN,2025,0,NaN,428224.0,1 A.M. Jailbreak,high,scored_candidate,5529.353000,25,False,False
4,1 A.M. Jailbreak,5189,10807,25EAST0410,Saashi & Saashi,NaN,2025,0,NaN,428224.0,1 A.M. Jailbreak,high,scored_candidate,5529.353000,25,False,False
5,10 days in asia,1,1593,TTL100RV,Out of the Box,NaN,2014,1,NaN,22398.0,10 Days in Asia,high,scored_candidate,2305.160000,25,False,False
6,1001 Islands,4036,8608,TTL16795I,Ludonaute,NaN,2022,0,NaN,351972.0,1001 Islands,high,scored_candidate,8770.136000,7,False,False
7,12 Days,5371,11144,25UNPL0061,Calliope Games,NaN,2025,1,NaN,130899.0,12 Days,high,scored_candidate,5221.258000,26,False,False
8,12 Rivers,4341,9137,TTL32241I,Dofinn,NaN,2022,1,NaN,370391.0,12 Rivers,manual,manual_title_fix,51229.699000,25,False,False
9,12 Thieves,1906,4026,TTL05202I,Queen Games,NaN,2017,1,NaN,22278.0,12 Thieves,high,scored_candidate,5980.326000,25,False,False


In [13]:
if REVIEW_PATH.exists():
    df_review = pd.read_csv(REVIEW_PATH)
    print(f"Review queue rows: {len(df_review):,}")
    display(df_review.head(50))
else:
    df_review = pd.DataFrame()
    print(f"Review queue not found: {REVIEW_PATH}")

Review queue rows: 523


,title_id,game_id,game_barcode,title,publisher_name,game_year,original_bgg_id,new_bgg_id,match_confidence,match_reason,match_score_margin,candidate_1_id,candidate_1_name,candidate_1_score,candidate_1_dimensions,candidate_1_title_score,candidate_1_publisher_match_type,candidate_1_year_match,candidate_1_id_match,candidate_1_pax_prior,candidate_1_is_expansion,candidate_1_has_promo_word,candidate_1_from_publisher_first_fallback,candidate_2_id,candidate_2_name,candidate_2_score,candidate_2_dimensions,candidate_2_title_score,candidate_2_publisher_match_type,candidate_2_year_match,candidate_2_id_match,candidate_2_pax_prior,candidate_2_is_expansion,candidate_2_has_promo_word,candidate_2_from_publisher_first_fallback,candidate_3_id,candidate_3_name,candidate_3_score,candidate_3_dimensions,candidate_3_title_score,candidate_3_publisher_match_type,candidate_3_year_match,candidate_3_id_match,candidate_3_pax_prior,candidate_3_is_expansion,candidate_3_has_promo_word,candidate_3_from_publisher_first_fallback
0,4,763,TTL1004W,3:16 Carnage Amogst the Stars,Box Ninja,2014,NaN,28843.0,low,scored_candidate,649.880000,28843.0,300: The Board Game,951.192,1.0,85.500,none,True,False,False,False,False,False,4921.0,3rd Fleet: Modern Naval Combat in the North Pa...,301.312,1.0,85.500,none,True,False,False,False,False,False,391260.0,3 & 4: The Game,301.000,1.0,85.500,none,True,False,False,False,False,False
1,3801,8137,TTL10109I,4 1/2 Tatami Mats,Hallelujah Rockboy,2021,NaN,187104.0,low,scored_candidate,0.899000,187104.0,4 Gods,1202.005,1.0,85.500,none,True,False,False,False,False,False,175263.0,4 the Birds,1201.106,1.0,85.500,none,True,False,False,False,False,False,213304.0,4 Seasons,1201.074,1.0,85.500,none,True,False,False,False,False,False
2,3801,8138,TTL10105I,4 1/2 Tatami Mats,Hallelujah Rockboy,2021,NaN,187104.0,low,scored_candidate,0.899000,187104.0,4 Gods,1202.005,1.0,85.500,none,True,False,False,False,False,False,175263.0,4 the Birds,1201.106,1.0,85.500,none,True,False,False,False,False,False,213304.0,4 Seasons,1201.074,1.0,85.500,none,True,False,False,False,False,False
3,1291,2951,TTL02693I,6 nimmt!,Mayfair Games,2015,432.0,432.0,low,scored_candidate,4118.646000,432.0,Beat the Heat,6334.420,3.0,28.500,any_exact,True,True,True,False,False,False,128011.0,Extra! Extra!,2215.774,2.0,42.750,first_exact,True,False,False,False,False,True,150930.0,Mad City,2205.304,2.0,37.500,first_exact,True,False,False,False,False,True
4,2162,4567,TTL03644I,7 Wonders Cities,Repos Production,2018,NaN,68448.0,low,scored_candidate,34.744000,68448.0,7 Wonders,3172.108,3.0,90.000,first_exact,True,False,False,False,False,False,111661.0,7 Wonders: Cities,3137.364,3.0,98.000,first_exact,True,False,False,True,False,False,173346.0,7 Wonders Duel,2386.645,2.0,74.348,first_exact,True,False,False,False,False,True
5,2162,4568,TTL03640I,7 Wonders Cities,Repos Production,2018,NaN,68448.0,low,scored_candidate,34.744000,68448.0,7 Wonders,3172.108,3.0,90.000,first_exact,True,False,False,False,False,False,111661.0,7 Wonders: Cities,3137.364,3.0,98.000,first_exact,True,False,False,True,False,False,173346.0,7 Wonders Duel,2386.645,2.0,74.348,first_exact,True,False,False,False,False,True
6,2162,4569,TTL03636I,7 Wonders Cities,Repos Production,2018,NaN,68448.0,low,scored_candidate,34.744000,68448.0,7 Wonders,3172.108,3.0,90.000,first_exact,True,False,False,False,False,False,111661.0,7 Wonders: Cities,3137.364,3.0,98.000,first_exact,True,False,False,True,False,False,173346.0,7 Wonders Duel,2386.645,2.0,74.348,first_exact,True,False,False,False,False,True
7,2162,4570,TTL03632I,7 Wonders Cities,Repos Production,2018,NaN,68448.0,low,scored_candidate,34.744000,68448.0,7 Wonders,3172.108,3.0,90.000,first_exact,True,False,False,False,False,False,111661.0,7 Wonders: Cities,3137.364,3.0,98.000,first_exact,True,False,False,True,False,False,173346.0,7 Wonders Duel,2386.645,2.0,74.348,first_exact,True,False,False,False,False,True
8,2639,5394,TTL05502I,A Game Of Folklore,Nibcard,2018,NaN,268678.0,low,pub

## Review finalized library output

The finalizer applies the confidence/status policy and optional manual fixes to produce the cleaned library export. In this run it produced 11,418 rows, including 6,199 BGG-ID updates and 1,289 title updates. This notebook only inspects that artifact; it does not reimplement the finalizer.

In [14]:
CLEAN_PATH = PAX_ROOT / "title_corrections_cleaned.csv"

if CLEAN_PATH.exists():
    df_clean = pd.read_csv(CLEAN_PATH)
    print(f"Cleaned rows: {len(df_clean):,}")
    print(f"Updated BGG IDs: {int(df_clean['has_updated_bgg_id'].sum()):,}")
    print(f"Updated titles: {int(df_clean['has_updated_title'].sum()):,}")
    display(df_clean.head(100))
else:
    df_clean = pd.DataFrame()
    print(f"Cleaned output not found: {CLEAN_PATH}")

Cleaned rows: 11,418
Updated BGG IDs: 6,199
Updated titles: 1,289


,title,title_id,game_id,game_barcode,bgg_id,match,has_updated_title,title_original,has_updated_bgg_id,bgg_id_original
0,"""A Matter of Honor, Sire""",1148,2680,TTL1019O,NaN,NaN,False,"""A Matter of Honor, Sire""",False,NaN
1,ARTBOX,5025,10440,TTL06494I,280304.0,92.31,True,#artbox,True,NaN
2,1 A.M. Jailbreak,5189,10805,25EAST0418,428224.0,100.00,False,1 A.M. Jailbreak,True,NaN
3,1 A.M. Jailbreak,5189,10806,25EAST0414,428224.0,100.00,False,1 A.M. Jailbreak,True,NaN
4,1 A.M. Jailbreak,5189,10807,25EAST0410,428224.0,100.00,False,1 A.M. Jailbreak,True,NaN
5,10 Days in Asia,1,1593,TTL100RV,22398.0,100.00,True,10 days in asia,True,NaN
6,1001 Islands,4036,8608,TTL16795I,351972.0,100.00,False,1001 Islands,True,NaN
7,12 Days,5371,11144,25UNPL0061,130899.0,100.00,False,12 Days,True,NaN
8,12 Rivers,4341,9137,TTL32241I,370391.0,100.00,False,12 Rivers,True,NaN
9,12 Thieves,1906,4026,TTL05202I,22278.0,100.00,False,12 Thieves,True,NaN


## Optional title-name spot checks

Use this section for targeted manual review of title replacements. It compares the cleaned title, original title, and resolver's proposed BGG name without changing the data.

In [15]:
if not df_clean.empty and not df_match.empty:
    df_tmp = (
        df_clean[["title", "title_id", "match", "bgg_id", "title_original"]]
        .dropna()
        .drop_duplicates()
        .merge(
            df_match[["title_id", "new_bgg_name"]].drop_duplicates(),
            on="title_id",
            how="left",
        )
    )
    display(df_tmp.sort_values("match", ascending=False).head(100))
else:
    df_tmp = pd.DataFrame()
    print("Load both resolver and cleaned outputs to run title-name spot checks.")

,title,title_id,match,bgg_id,title_original,new_bgg_name
4559,Zombie Tower 3D,3129,100.0,189829.0,Zombie Tower 3d,Zombie Tower 3D
4558,Zombie Tower 3D,2200,100.0,189829.0,Zombie Tower 3d,Zombie Tower 3D
4557,Zombie Tower 3D,1331,100.0,189829.0,Zombie Tower 3D,Zombie Tower 3D
4556,Zombie Survival: The Board Game,1232,100.0,79312.0,Zombie Survival: The Board Game,Zombie Survival: The Board Game
4555,Zombie Slam,3194,100.0,231673.0,Zombie Slam,Zombie Slam
4554,Zombie Princess and the Enchanted Maze,3650,100.0,333116.0,Zombie Princess and the Enchanted Maze,Zombie Princess and the Enchanted Maze
4553,Zombie Ninja Pirates,983,100.0,63067.0,Zombie Ninja Pirates,Zombie Ninja Pirates
4552,Zombie Mosh!,982,100.0,39789.0,Zombie Mosh!,Zombie Mosh!
4550,Zombie Fluxx,980,100.0,29387.0,Zombie Fluxx,Zombie Fluxx
4548,Zombie Dice,979,100.0,62871.0,Zombie Dice,Zombie Dice


## Export the East library BGG-ID list

This optional export is intentionally narrow: one BGG ID per eligible title. It is generated from resolver output, not from the manual mapping table. Review the resolver queue before using it downstream.

In [ ]:
EAST_EXPORT_PATH = PAX_ROOT / "pax_east_2026_library.csv"

if not df_match.empty:
    east_bgg_ids = (
        df_match.loc[
            (df_match["status"] == 0) & df_match["new_bgg_id"].notna(),
            ["new_bgg_id"],
        ]
        .drop_duplicates()
    )
    east_bgg_ids.to_csv(EAST_EXPORT_PATH, index=False)
    print(f"Wrote {len(east_bgg_ids):,} BGG IDs to {EAST_EXPORT_PATH}")
else:
    print("Resolver output is required before creating the East export.")